# 016 — Diseño y validación de heurísticas

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("search", seed=16)
assert result["kind"] == "search"
assert result["evidence"]
show(result)


## Solución 1 — h1 y h2

a) Mal colocadas: `7` (está en (2,1), debe estar en (2,0)) y `8` (está en (2,2),
debe estar en (2,1)) → **h1 = 2**.

b) Manhattan: ficha 7: `|2-2| + |1-0| = 1`; ficha 8: `|2-2| + |2-1| = 1`;
el resto aporta 0 → **h2 = 2**.

c) Solución real: deslizar 7 a la izquierda y luego 8 a la izquierda → costo
real `h* = 2`. Aquí ambas heurísticas son exactas; en estados más revueltos
`h2 > h1` y ambas quedan por debajo de `h*`.


In [ ]:
estado = [[1, 2, 3], [4, 5, 6], [0, 7, 8]]
objetivo = [[1, 2, 3], [4, 5, 6], [7, 8, 0]]
pos_meta = {objetivo[f][c]: (f, c) for f in range(3) for c in range(3)}
h1 = sum(1 for f in range(3) for c in range(3)
         if estado[f][c] != 0 and estado[f][c] != objetivo[f][c])
h2 = sum(abs(f - pos_meta[v][0]) + abs(c - pos_meta[v][1])
         for f in range(3) for c in range(3) if (v := estado[f][c]) != 0)
assert (h1, h2) == (2, 2)
print(f"h1={h1}, h2={h2} ✔")


## Solución 2 — Dominancia y admisibilidad

a) Toda ficha mal colocada está al menos a 1 de Manhattan de su sitio, así que
cada unidad de `h1` aporta ≥ 1 a `h2`: `h2 ≥ h1` en todo estado.

b) `h3 = 3·h2` **no es admisible**: en el estado del ejercicio 1, `h3 = 6`
pero el costo real es 2 — sobreestima. A* con `h3` podría devolver una
solución subóptima.

c) Como `h2 ≥ h1` siempre, `max(h1, h2) = h2`: el máximo no aporta nada aquí.
La combinación por máximo es útil cuando las heurísticas son **incomparables**
(cada una gana en estados distintos).


## Solución 3 — Diseño por relajación

a) Relajación: **eliminar los muros**. El problema relajado se resuelve exacto
con `|Δfila| + |Δcolumna|` → Manhattan, admisible por construcción.

b) Otra relajación: **permitir movimiento diagonal** (además de quitar muros)
→ distancia de Chebyshev `max(|Δfila|, |Δcolumna|)`.

c) Chebyshev ≤ Manhattan siempre, así que **Manhattan domina**: relajar *más*
(dos restricciones en vez de una) da estimaciones más flojas. Regla general:
la relajación mínima que siga siendo tratable produce la mejor heurística.


## Solución 4 — Heurística para el grafo del laboratorio

a) BFS expande los 7 nodos: `[A, B, C, D, E, F, G]`.

b) Distancias reales a G (en aristas): `h(G)=0, h(E)=1, h(F)=1, h(B)=2,
h(C)=2, h(A)=3, h(D)=∞` (D no alcanza G). Con esa `h` exacta, A* (f = g+h)
mantiene f=3 a lo largo del camino óptimo y expande `A, B, C, E, F, G` o
incluso menos según desempates — pero nunca `D`, cuyo `f` es infinito. El
ahorro concreto de la información heurística: podar las ramas sin salida.


In [ ]:
result = run_lab("search", seed=16)
assert "D" in result["result"]["expanded"]
print("BFS expandió D; A* con h informada lo habría podado ✔")


## Reflexión

1. ¿Por qué la técnica de 'relajar el problema' garantiza admisibilidad por construcción, mientras que una heurística inventada 'a ojo' debe verificarse caso por caso?
2. Si una heurística inadmisible acelera la búsqueda pero pierde la garantía de optimalidad, ¿en qué tipo de aplicación aceptarías ese intercambio y en cuál no? Da un ejemplo concreto de cada una.
3. h=0 es admisible y h=h* es la heurística perfecta. ¿Qué le pasa a A* en cada extremo y qué te dice eso sobre el 'precio' de la información?
